# Simple integration of LLM and LangGraph
##### start -> llm_question_answer -> end

In [ ]:
from langgraph.graph import StateGraph,START,END
from langchain_huggingface import ChatHuggingFace,HuggingFaceEndpoint
from dotenv import load_dotenv
from typing import TypedDict

In [ ]:
load_dotenv()
llm = HuggingFaceEndpoint(
    repo_id="openai/gpt-oss-20b",
    task="text-generation"
)
model = ChatHuggingFace(llm = llm)

/media/puskarkumar/Study&Proj/webDevelopment/Projects/Study/LangGraph/virtEnv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
#create a state
class LLMState(TypedDict):
    
    question: str
    answer: str 


In [ ]:
def llm_qa(state: LLMState) -> LLMState:
    
    #extract the question from state 
    question = state['question']
    
    #form the prompt
    prompt = f"answer the following question {question}"
    
    #ask that question to the LLM
    answer = model.invoke(prompt).content
    
    #update the answer in the state
    state['answer'] = answer
    
    return state

In [ ]:
#create our graph 
graph = StateGraph(LLMState)

#add nodes 
graph.add_node('llm_qa',llm_qa)

#add edges 
graph.add_edge(START,'llm_qa')
graph.add_edge('llm_qa',END)

#compile 
workflow = graph.compile()

In [ ]:
#evaluate the workflow
initial_state = {'question':'How far is moon from earth ?'}
final_state = workflow.invoke(initial_state)
print(final_state)

{'question': 'How far is moon from earth ?', 'answer': 'The average distance from Earth to the Moon is about **384,400\u202fkm** (≈\u202f238,855\u202fmiles).  \n\nBecause the Moon’s orbit is elliptical, the distance varies:\n\n- **Perigee (closest point)**: ~\u202f363,300\u202fkm (≈\u202f225,623\u202fmi)  \n- **Apogee (farthest point)**: ~\u202f405,500\u202fkm (≈\u202f251,968\u202fmi)  \n\nSo the Moon is roughly 384,400\u202fkm away on average, but it can be a bit closer or farther depending on where it is in its orbit.'}
